# Indian Mutual Fund Portfolio Overlap Analyzer
### By Daksh Malhotra

**Objective:** Millions of Indian investors hold 3-5 mutual funds thinking they're diversified, but most funds hold the same stocks. This project analyzes portfolio overlap across 45 top Indian equity mutual funds to uncover hidden concentration risks.

**Data Source:** Scraped from Moneycontrol.com (publicly available portfolio holdings)

**Tools:** Python (requests, BeautifulSoup, pandas), SQL (SQLite)

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import json

## Data Collection
I built a web scraper to extract stock holdings from Moneycontrol. Each fund's portfolio page has a table (`equityCompleteHoldingTable`) containing stock names, sectors, holding percentages, and other details.

**Key Challenges:**
1. Moneycontrol hides exited/past holdings using `display: none` in the HTML — I had to detect and filter these out to get only current holdings.
2. Some table rows were empty, containing only symbols like `-` and `#` — filtered these out using a length check on stock names.

In [2]:
def scrape_fund_holdings(fund_name, url):

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to fetch {fund_name}")
        return []
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.find('table', {'id': 'equityCompleteHoldingTable'})
    
    if not table:
        print(f"No holdings table found for {fund_name}")
        return []
    
    rows = table.find('tbody').find_all('tr')
    holdings = []
    
    for row in rows:
        if row.get('style') == 'display: none;':  # Moneycontrol hides exited stocks with display:none — skip these
            continue
            
        cols = row.find_all('td')
        if len(cols) >= 11:
            stock_name = cols[0].text.strip()
            if len(stock_name) > 2: # Some rows have junk characters like '-' or '#' instead of stock names
                holdings.append({
                    'fund_name': fund_name,
                    'stock_name': stock_name,
                    'sector': cols[1].text.strip(),
                    'holding_pct': cols[4].text.strip().replace('%', ''),
                    'market_cap': cols[10].text.strip()
                })
    
    print(f"✓ {fund_name}: {len(holdings)} stocks found")
    return holdings

## Fund Selection
I selected the **top 10 funds by AUM** in each of the following categories:
- Large Cap (10 funds)
- Mid Cap (10 funds)  
- Small Cap (10 funds)
- Flexi Cap (10 funds)
- Index Funds (5 funds — Nifty 50 + Sensex trackers)

**Total: 45 funds** representing the most popular equity mutual funds in India.

In [3]:
funds = {
    # LARGE CAP (10)
    'ICICI Prudential Large Cap Fund': 'https://www.moneycontrol.com/mutual-funds/icici-prudential-large-cap-fund-direct-plan/portfolio-holdings/MPI1134',
    'SBI Large Cap Fund': 'https://www.moneycontrol.com/mutual-funds/sbi-large-cap-fund-direct-plan/portfolio-holdings/MSB532',
    'Nippon India Large Cap Fund': 'https://www.moneycontrol.com/mutual-funds/nippon-india-large-cap-fund-direct-plan/portfolio-holdings/MRC940',
    'Mirae Asset Large Cap Fund': 'https://www.moneycontrol.com/mutual-funds/mirae-asset-large-cap-fund-direct-plan/portfolio-holdings/MMA096',
    'HDFC Large Cap Fund': 'https://www.moneycontrol.com/mutual-funds/hdfc-large-cap-fund-direct-plan-growth/portfolio-holdings/MHD1169',
    'Axis Large Cap Fund': 'https://www.moneycontrol.com/mutual-funds/axis-large-cap-fund-direct-plan/portfolio-holdings/MAA181',
    'Aditya Birla Sun Life Large Cap Fund': 'https://www.moneycontrol.com/mutual-funds/aditya-birla-sun-life-large-cap-fund-direct-plan/portfolio-holdings/MBS813',
    'Canara Robeco Large Cap Fund': 'https://www.moneycontrol.com/mutual-funds/canara-robeco-large-cap-fund-direct-plan/portfolio-holdings/MCA212',
    'UTI Large Cap Fund': 'https://www.moneycontrol.com/mutual-funds/uti-large-cap-fund-direct-plan-growth/portfolio-holdings/MUT639',
    'Kotak Large Cap Fund': 'https://www.moneycontrol.com/mutual-funds/kotak-bluechip-fund-direct-plan/portfolio-holdings/MKM514',

    # MID CAP (10)
    'HDFC Mid Cap Fund': 'https://www.moneycontrol.com/mutual-funds/hdfc-mid-cap-opportunities-fund-direct-plan/portfolio-holdings/MHD1161',
    'Kotak Midcap Fund': 'https://www.moneycontrol.com/mutual-funds/kotak-midcap-fund-direct-plan/portfolio-holdings/MKM528',
    'Nippon India Growth Mid Cap Fund': 'https://www.moneycontrol.com/mutual-funds/nippon-india-growth-fund-direct-plan/portfolio-holdings/MRC919',
    'Motilal Oswal Midcap Fund': 'https://www.moneycontrol.com/mutual-funds/motilal-oswal-midcap-fund-direct-plan/portfolio-holdings/MMO027',
    'Axis Midcap Fund': 'https://www.moneycontrol.com/mutual-funds/axis-mid-cap-fund-direct-plan/portfolio-holdings/MAA194',
    'SBI Midcap Fund': 'https://www.moneycontrol.com/mutual-funds/sbi-midcap-fund-direct-plan/portfolio-holdings/MSB505',
    'DSP Midcap Fund': 'https://www.moneycontrol.com/mutual-funds/dsp-blackrock-mid-cap-fund-direct-plan/portfolio-holdings/MDS574',
    'Mirae Asset Midcap Fund': 'https://www.moneycontrol.com/mutual-funds/mirae-asset-midcap-fund-direct-plan/portfolio-holdings/MMA173',
    'Edelweiss Mid Cap Fund': 'https://www.moneycontrol.com/mutual-funds/edelweiss-mid-cap-fund-direct-plan/portfolio-holdings/MJP117',
    'Sundaram Mid Cap Fund': 'https://www.moneycontrol.com/mutual-funds/sundaram-mid-cap-fund-direct-plan/portfolio-holdings/MSN568',

    # SMALL CAP (10)
    'Nippon India Small Cap Fund': 'https://www.moneycontrol.com/mutual-funds/nippon-india-small-cap-fund-direct-plan/portfolio-holdings/MRC935',
    'HDFC Small Cap Fund': 'https://www.moneycontrol.com/mutual-funds/hdfc-small-cap-fund-direct-plan/portfolio-holdings/MMS025',
    'SBI Small Cap Fund': 'https://www.moneycontrol.com/mutual-funds/sbi-small-cap-fund-direct-plan-/portfolio-holdings/MSA031',
    'Quant Small Cap Fund': 'https://www.moneycontrol.com/mutual-funds/quant-small-cap-fund-direct-plan/portfolio-holdings/MES056',
    'Axis Small Cap Fund': 'https://www.moneycontrol.com/mutual-funds/axis-small-cap-fund-direct-plan/portfolio-holdings/MAA316',
    'Bandhan Small Cap Fund': 'https://www.moneycontrol.com/mutual-funds/bandhan-small-cap-fund-direct-plan-growth/portfolio-holdings/MAG2104',
    'DSP Small Cap Fund': 'https://www.moneycontrol.com/mutual-funds/dsp-small-cap-fund-direct-plan/portfolio-holdings/MDS584',
    'Kotak Small Cap Fund': 'https://www.moneycontrol.com/mutual-funds/kotak-small-cap-fund-direct-plan/portfolio-holdings/MKM516',
    'HSBC Small Cap Fund': 'https://www.moneycontrol.com/mutual-funds/hsbc-small-cap-fund-direct-plan/portfolio-holdings/MCC492',
    'Franklin India Small Cap Fund': 'https://www.moneycontrol.com/mutual-funds/franklin-india-smaller-companies-fund-direct-plan/portfolio-holdings/MTE313',

    # FLEXI CAP (10)
    'Parag Parikh Flexi Cap Fund': 'https://www.moneycontrol.com/mutual-funds/parag-parikh-flexi-cap-fund-direct-plan/portfolio-holdings/MPP002',
    'HDFC Flexi Cap Fund': 'https://www.moneycontrol.com/mutual-funds/hdfc-flexi-cap-fund-direct-plan-/portfolio-holdings/MHD1144',
    'Kotak Flexi Cap Fund': 'https://www.moneycontrol.com/mutual-funds/kotak-flexicap-fund-direct-plan/portfolio-holdings/MKM520',
    'Aditya Birla Sun Life Flexi Cap Fund': 'https://www.moneycontrol.com/mutual-funds/aditya-birla-sun-life-flexi-cap-fund-direct-plan/portfolio-holdings/MBS811',
    'SBI Flexi Cap Fund': 'https://www.moneycontrol.com/mutual-funds/sbi-flexicap-fund-direct-plan/portfolio-holdings/MSB503',
    'UTI Flexi Cap Fund': 'https://www.moneycontrol.com/mutual-funds/uti-flexi-cap-fund-direct-plan/portfolio-holdings/MUT651',
    'ICICI Prudential Flexicap Fund': 'https://www.moneycontrol.com/mutual-funds/icici-prudential-flexicap-fund-direct-plan/portfolio-holdings/MPI4506',
    'Franklin India Flexi Cap Fund': 'https://www.moneycontrol.com/mutual-funds/franklin-india-flexi-cap-fund-direct-plan/portfolio-holdings/MTE315',
    'Canara Robeco Flexi Cap Fund': 'https://www.moneycontrol.com/mutual-funds/canara-robeco-flexi-cap-fund-direct-plan/portfolio-holdings/MCA202',
    'Motilal Oswal Flexi Cap Fund': 'https://www.moneycontrol.com/mutual-funds/motilal-oswal-flexi-cap-fund-direct-plan/portfolio-holdings/MMO031',

    # INDEX FUNDS (5)
    'UTI Nifty 50 Index Fund': 'https://www.moneycontrol.com/mutual-funds/uti-nifty-50-index-fund-direct-plan/portfolio-holdings/MUT657',
    'HDFC Nifty 50 Index Fund': 'https://www.moneycontrol.com/mutual-funds/hdfc-index-fund-nifty-50-plan-direct-plan/portfolio-holdings/MHD1152',
    'ICICI Prudential Nifty 50 Index Fund': 'https://www.moneycontrol.com/mutual-funds/icici-prudential-nifty-50-index-fund-direct-plan/portfolio-holdings/MPI1144',
    'SBI Nifty Index Fund': 'https://www.moneycontrol.com/mutual-funds/sbi-nifty-index-fund-direct-plan/portfolio-holdings/MSB507',
    'HDFC BSE Sensex Index Fund': 'https://www.moneycontrol.com/mutual-funds/hdfc-index-fund-bse-sensex-plan-direct-plan/portfolio-holdings/MHD1153',
}

print(f"Total funds: {len(funds)}")

Total funds: 45


In [4]:
all_holdings = []
success = 0
failed = []

for fund_name, url in funds.items():
    holdings = scrape_fund_holdings(fund_name, url)
    if holdings:
        all_holdings.extend(holdings)
        success += 1
    else:
        failed.append(fund_name)
        # Polite scraping - 2 second gap between requests so we don't overload Moneycontrol
    time.sleep(2)

print(f"\nScraping complete! {success}/45 funds scraped, {len(all_holdings)} total holdings")

✓ ICICI Prudential Large Cap Fund: 69 stocks found
✓ SBI Large Cap Fund: 45 stocks found
✓ Nippon India Large Cap Fund: 67 stocks found
✓ Mirae Asset Large Cap Fund: 79 stocks found
✓ HDFC Large Cap Fund: 46 stocks found
✓ Axis Large Cap Fund: 51 stocks found
✓ Aditya Birla Sun Life Large Cap Fund: 77 stocks found
✓ Canara Robeco Large Cap Fund: 60 stocks found
✓ UTI Large Cap Fund: 61 stocks found
✓ Kotak Large Cap Fund: 60 stocks found
✓ HDFC Mid Cap Fund: 78 stocks found
✓ Kotak Midcap Fund: 65 stocks found
✓ Nippon India Growth Mid Cap Fund: 99 stocks found
✓ Motilal Oswal Midcap Fund: 26 stocks found
✓ Axis Midcap Fund: 96 stocks found
✓ SBI Midcap Fund: 53 stocks found
✓ DSP Midcap Fund: 63 stocks found
✓ Mirae Asset Midcap Fund: 64 stocks found
✓ Edelweiss Mid Cap Fund: 88 stocks found
✓ Sundaram Mid Cap Fund: 78 stocks found
✓ Nippon India Small Cap Fund: 244 stocks found
✓ HDFC Small Cap Fund: 84 stocks found
✓ SBI Small Cap Fund: 67 stocks found
✓ Quant Small Cap Fund: 103 st

## Database Design
Loaded all holdings into a SQLite database for SQL analysis. Each row represents one stock holding in one fund.

**Schema:** `holdings(fund_name, stock_name, sector, holding_pct, market_cap)`

In [13]:
df = pd.DataFrame(all_holdings)
df['holding_pct'] = pd.to_numeric(df['holding_pct'], errors='coerce')

# saving to sqlite
conn = sqlite3.connect('mutual_fund_overlap.db')
df.to_sql('holdings', conn, if_exists='replace', index=False)
df.to_csv('mutual_fund_holdings.csv', index=False)

print(f"Database: {len(df)} rows, {df['fund_name'].nunique()} funds, {df['stock_name'].nunique()} unique stocks")

Database: 3421 rows, 45 funds, 1041 unique stocks


In [14]:
df.head()

,fund_name,stock_name,sector,holding_pct,market_cap
0,ICICI Prudential Large Cap Fund,ICICI Bank Ltd.,Private sector bank,9.42,Large Cap
1,ICICI Prudential Large Cap Fund,HDFC Bank Ltd.,Private sector bank,9.16,Other
2,ICICI Prudential Large Cap Fund,Larsen & Toubro Ltd.,Civil construction,6.34,Large Cap
3,ICICI Prudential Large Cap Fund,Reliance Industries Ltd.,Refineries & marketing,5.98,Large Cap
4,ICICI Prudential Large Cap Fund,-\nAxis Bank Ltd.,Private sector bank,4.61,Large Cap


## Backup
Saving data in multiple formats for reproducibility.

In [6]:
# Saving the cleaned data as CSV for backup
df = pd.DataFrame(all_holdings)
df.to_csv('mutual_fund_holdings.csv', index=False)

# Also saving the funds dictionary for reference
import json
with open('funds_list.json', 'w') as f:
    json.dump(funds, f, indent=2)

print("Data saved!")
print(f"  mutual_fund_overlap.db (SQLite database)")
print(f"  mutual_fund_holdings.csv (backup CSV)")
print(f"  funds_list.json (fund URLs)")

Data saved!
  mutual_fund_overlap.db (SQLite database)
  mutual_fund_holdings.csv (backup CSV)
  funds_list.json (fund URLs)


### Which stocks appear in the most mutual funds?
If the same stock is held by 30+ funds, any investor holding multiple funds is unknowingly over-exposed to that stock. Let's find out which stocks dominate Indian mutual fund portfolios.

In [7]:
query = """
SELECT 
    stock_name,
    COUNT(DISTINCT fund_name) as num_funds,
    ROUND(AVG(holding_pct), 2) as avg_holding_pct
FROM holdings
WHERE holding_pct > 0
GROUP BY stock_name
HAVING num_funds > 1
ORDER BY num_funds DESC
LIMIT 20
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                             stock_name  num_funds  avg_holding_pct
                           Eternal Ltd.         31             1.95
                        ICICI Bank Ltd.         28             6.94
                         HDFC Bank Ltd.         28             7.54
                         Axis Bank Ltd.         26             3.24
               Reliance Industries Ltd.         24             5.23
                   Larsen & Toubro Ltd.         24             3.93
               Kotak Mahindra Bank Ltd.         24             2.86
                           Infosys Ltd.         24             3.31
                     Bharti Airtel Ltd.         24             3.72
                    State Bank Of India         23             3.69
               Maruti Suzuki India Ltd.         22             2.24
     Sun Pharmaceutical Industries Ltd.         19             1.40
National Thermal Power Corporation Ltd.         19             1.63
               Mahindra & Mahindra Ltd.         

### Which fund pairs have the highest portfolio overlap?
This is the core question — if I hold Fund A and Fund B, how similar are their portfolios? I'm using weighted overlap (sum of minimum weights of common stocks), which is the same formula used by professional tools like Dezerv and PrimeInvestor.

In [8]:
query = """
WITH fund_pairs AS (
    SELECT 
        a.fund_name as fund_1,
        b.fund_name as fund_2,
        COUNT(DISTINCT a.stock_name) as common_stocks,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as weighted_overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name < b.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
    GROUP BY a.fund_name, b.fund_name
)
SELECT 
    fund_1,
    fund_2,
    common_stocks,
    weighted_overlap
FROM fund_pairs
ORDER BY weighted_overlap DESC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                              fund_1                               fund_2  common_stocks  weighted_overlap
            HDFC Nifty 50 Index Fund                 SBI Nifty Index Fund             50            100.00
            HDFC Nifty 50 Index Fund              UTI Nifty 50 Index Fund             50            100.00
                SBI Nifty Index Fund              UTI Nifty 50 Index Fund             50            100.00
            HDFC Nifty 50 Index Fund ICICI Prudential Nifty 50 Index Fund             49             98.38
ICICI Prudential Nifty 50 Index Fund                 SBI Nifty Index Fund             49             98.38
ICICI Prudential Nifty 50 Index Fund              UTI Nifty 50 Index Fund             49             98.38
          HDFC BSE Sensex Index Fund ICICI Prudential Nifty 50 Index Fund             30             83.53
          HDFC BSE Sensex Index Fund             HDFC Nifty 50 Index Fund             29             82.34
          HDFC BSE Sensex Index Fund 

### Which fund pairs have the LEAST overlap?
The flip side — these are the best fund combinations for diversification. If two funds share almost no stocks, holding both gives you genuine diversification.

In [9]:
query = """
WITH fund_pairs AS (
    SELECT 
        a.fund_name as fund_1,
        b.fund_name as fund_2,
        COUNT(DISTINCT a.stock_name) as common_stocks,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as weighted_overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name < b.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
    GROUP BY a.fund_name, b.fund_name
)
SELECT 
    fund_1,
    fund_2,
    common_stocks,
    weighted_overlap
FROM fund_pairs
ORDER BY weighted_overlap ASC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                       fund_1                          fund_2  common_stocks  weighted_overlap
  Parag Parikh Flexi Cap Fund           Sundaram Mid Cap Fund              3              0.09
            Kotak Midcap Fund     Parag Parikh Flexi Cap Fund              1              0.12
         Quant Small Cap Fund                 SBI Midcap Fund              1              0.14
           DSP Small Cap Fund     Parag Parikh Flexi Cap Fund              1              0.27
          HDFC Small Cap Fund     Parag Parikh Flexi Cap Fund              2              0.32
          HDFC Flexi Cap Fund             HSBC Small Cap Fund              1              0.33
          HSBC Small Cap Fund ICICI Prudential Large Cap Fund              1              0.34
       Bandhan Small Cap Fund    Motilal Oswal Flexi Cap Fund              2              0.38
 Canara Robeco Large Cap Fund             HSBC Small Cap Fund              1              0.38
 Canara Robeco Flexi Cap Fund              SBI Sma

### Most "crowded" stocks — where is mutual fund money concentrated?
Beyond just counting appearances, I wanted to see which stocks have the most total money flowing into them across all funds. A stock held by 25 funds at 7% each represents massive concentration risk for the market.

In [12]:
query = """
SELECT 
    stock_name,
    COUNT(DISTINCT fund_name) as num_funds,
    ROUND(AVG(holding_pct), 2) as avg_weight,
    ROUND(MAX(holding_pct), 2) as max_weight,
    ROUND(SUM(holding_pct), 2) as total_weight_across_funds
FROM holdings
WHERE holding_pct > 0
GROUP BY stock_name
ORDER BY total_weight_across_funds DESC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                             stock_name  num_funds  avg_weight  max_weight  total_weight_across_funds
                         HDFC Bank Ltd.         28        7.54       14.07                     211.21
                        ICICI Bank Ltd.         28        6.94       10.27                     194.21
               Reliance Industries Ltd.         24        5.23        9.83                     125.49
                   Larsen & Toubro Ltd.         24        3.93        6.34                      94.25
                     Bharti Airtel Ltd.         24        3.72        5.59                      89.29
                    State Bank Of India         23        3.69        5.26                      84.81
                         Axis Bank Ltd.         26        3.24        7.44                      84.25
                           Infosys Ltd.         24        3.31        4.88                      79.50
               Kotak Mahindra Bank Ltd.         24        2.86        5.34        

### Which sectors dominate Indian mutual fund portfolios?
If most funds are overweight on the same sectors, a sector-specific downturn (like the 2018 NBFC crisis) would hit almost every mutual fund investor simultaneously.

In [13]:
query = """
SELECT 
    sector,
    COUNT(DISTINCT stock_name) as unique_stocks,
    COUNT(DISTINCT fund_name) as funds_invested,
    ROUND(AVG(holding_pct), 2) as avg_weight,
    ROUND(SUM(holding_pct), 2) as total_weight
FROM holdings
WHERE holding_pct > 0
GROUP BY sector
ORDER BY total_weight DESC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                                  sector  unique_stocks  funds_invested  avg_weight  total_weight
                     Private sector bank             23              45        3.89        665.63
       Computers - software & consulting             24              45        1.52        240.47
                         Pharmaceuticals             68              42        0.94        205.81
            Auto components & equipments             53              36        0.97        153.99
                  Refineries & marketing              7              35        3.27        147.15
    Non banking financial company (nbfc)             26              42        1.42        145.24
                      Civil construction             27              37        1.64        133.20
                      Public sector bank             11              36        2.54        121.96
       Passenger cars & utility vehicles              6              25        1.95        112.93
Telecom - cellular &

### Which funds are most concentrated (least diversified internally)?
Fewer stocks = higher conviction but higher risk. I wanted to see which funds run concentrated portfolios.

In [30]:
query = """
SELECT 
    fund_name,
    COUNT(stock_name) as total_stocks,
    ROUND(SUM(CASE WHEN holding_pct >= 5 THEN holding_pct ELSE 0 END), 2) as weight_in_top_holdings,
    ROUND(MAX(holding_pct), 2) as largest_single_stock,
    ROUND(AVG(holding_pct), 2) as avg_holding
FROM holdings
WHERE holding_pct > 0
GROUP BY fund_name
ORDER BY total_stocks ASC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                           fund_name  total_stocks  weight_in_top_holdings  largest_single_stock  avg_holding
        Motilal Oswal Flexi Cap Fund            23                   52.27                  8.93         4.02
           Motilal Oswal Midcap Fund            26                   56.19                  7.64         3.77
          HDFC BSE Sensex Index Fund            30                   50.05                 14.07         3.33
                  SBI Large Cap Fund            45                   26.65                  8.20         2.12
                 HDFC Large Cap Fund            46                   29.01                  9.50         2.14
            HDFC Nifty 50 Index Fund            50                   28.61                 11.83         2.00
ICICI Prudential Nifty 50 Index Fund            50                   28.54                 11.80         2.00
                SBI Nifty Index Fund            50                   28.61                 11.83         2.00
          

### Real-world scenario: What happens if I hold both HDFC Large Cap and ICICI Large Cap?
Many investors hold 2-3 large cap funds thinking they're diversified. Let's see the actual combined exposure — how much of your money ends up in the same stocks?

In [ ]:
query = """
SELECT 
    COALESCE(a.stock_name, b.stock_name) as stock_name,
    ROUND(COALESCE(a.holding_pct, 0), 2) as hdfc_large_cap,
    ROUND(COALESCE(b.holding_pct, 0), 2) as icici_large_cap,
    ROUND(COALESCE(a.holding_pct, 0) + COALESCE(b.holding_pct, 0), 2) as combined_exposure
FROM holdings a
FULL OUTER JOIN holdings b 
    ON a.stock_name = b.stock_name
    AND b.fund_name = 'ICICI Prudential Large Cap Fund'
WHERE a.fund_name = 'HDFC Large Cap Fund'
    AND (a.holding_pct > 0 OR b.holding_pct > 0)
ORDER BY combined_exposure DESC
LIMIT 15
"""

try:
    result = pd.read_sql_query(query, conn)
    print(result.to_string(index=False))
except:
    # SQLite doesn't support FULL OUTER JOIN, use alternative
    query = """
    SELECT 
        h1.stock_name,
        ROUND(h1.holding_pct, 2) as hdfc_large_cap,
        ROUND(COALESCE(h2.holding_pct, 0), 2) as icici_large_cap,
        ROUND(h1.holding_pct + COALESCE(h2.holding_pct, 0), 2) as combined_exposure
    FROM holdings h1
    LEFT JOIN holdings h2 
        ON h1.stock_name = h2.stock_name
        AND h2.fund_name = 'ICICI Prudential Large Cap Fund'
    WHERE h1.fund_name = 'HDFC Large Cap Fund'
        AND h1.holding_pct > 0
    ORDER BY combined_exposure DESC
    LIMIT 15
    """
    result = pd.read_sql_query(query, conn)
    print(result.to_string(index=False))

### Do large cap funds overlap more than small cap funds?
My hypothesis: Large cap funds should overlap more because SEBI mandates them to invest 80%+ in only ~100 stocks. Small cap funds have 1000+ options. Let's test this.

In [16]:
query = """
WITH fund_categories AS (
    SELECT fund_name,
        CASE 
            WHEN fund_name LIKE '%Large Cap%' THEN 'Large Cap'
            WHEN fund_name LIKE '%Mid Cap%' OR fund_name LIKE '%Midcap%' THEN 'Mid Cap'
            WHEN fund_name LIKE '%Small Cap%' THEN 'Small Cap'
            WHEN fund_name LIKE '%Flexi%' OR fund_name LIKE '%Flexicap%' THEN 'Flexi Cap'
            WHEN fund_name LIKE '%Index%' OR fund_name LIKE '%Sensex%' OR fund_name LIKE '%Nifty%' THEN 'Index'
            ELSE 'Other'
        END as category
    FROM holdings
    GROUP BY fund_name
),
overlap AS (
    SELECT 
        c1.category as category,
        a.fund_name as fund_1,
        b.fund_name as fund_2,
        COUNT(DISTINCT a.stock_name) as common_stocks,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as weighted_overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name < b.fund_name
    JOIN fund_categories c1 ON a.fund_name = c1.fund_name
    JOIN fund_categories c2 ON b.fund_name = c2.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
        AND c1.category = c2.category
    GROUP BY a.fund_name, b.fund_name
)
SELECT 
    category,
    COUNT(*) as fund_pairs,
    ROUND(AVG(common_stocks), 1) as avg_common_stocks,
    ROUND(AVG(weighted_overlap), 2) as avg_overlap_pct,
    ROUND(MAX(weighted_overlap), 2) as max_overlap_pct
FROM overlap
GROUP BY category
ORDER BY avg_overlap_pct DESC
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

 category  fund_pairs  avg_common_stocks  avg_overlap_pct  max_overlap_pct
    Index          10               41.4            92.57           100.00
Large Cap          45               26.7            54.42            72.71
Flexi Cap          45               14.3            30.24            46.67
  Mid Cap          44               21.9            25.24            43.56
Small Cap          45               18.8            11.14            25.41


### Zero overlap fund pairs — can two mid cap funds share absolutely no stocks?
While analyzing the category overlap data, I noticed one mid cap pair was missing from the results. This query finds which two funds have zero common stocks — despite being in the same category.

In [17]:
query = """
WITH midcap_funds AS (
    SELECT DISTINCT fund_name 
    FROM holdings 
    WHERE fund_name LIKE '%Mid Cap%' OR fund_name LIKE '%Midcap%'
),
all_pairs AS (
    SELECT a.fund_name as fund_1, b.fund_name as fund_2
    FROM midcap_funds a
    CROSS JOIN midcap_funds b
    WHERE a.fund_name < b.fund_name
),
overlap_pairs AS (
    SELECT 
        a.fund_name as fund_1, 
        b.fund_name as fund_2
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name < b.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
    GROUP BY a.fund_name, b.fund_name
)
SELECT ap.fund_1, ap.fund_2
FROM all_pairs ap
LEFT JOIN overlap_pairs op 
    ON ap.fund_1 = op.fund_1 AND ap.fund_2 = op.fund_2
WHERE op.fund_1 IS NULL
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                   fund_1          fund_2
Motilal Oswal Midcap Fund SBI Midcap Fund


### Which category combinations give the best diversification?
If I want to hold one fund from two different categories, which combination gives me the least overlap? This helps build an optimal multi-category portfolio.

In [18]:
query = """
WITH fund_cat AS (
    SELECT fund_name,
        CASE 
            WHEN fund_name LIKE '%Large Cap%' THEN 'Large Cap'
            WHEN fund_name LIKE '%Mid Cap%' OR fund_name LIKE '%Midcap%' THEN 'Mid Cap'
            WHEN fund_name LIKE '%Small Cap%' THEN 'Small Cap'
            WHEN fund_name LIKE '%Flexi%' OR fund_name LIKE '%Flexicap%' THEN 'Flexi Cap'
            WHEN fund_name LIKE '%Index%' OR fund_name LIKE '%Sensex%' OR fund_name LIKE '%Nifty%' THEN 'Index'
            ELSE 'Other'
        END as category
    FROM holdings
    GROUP BY fund_name
),
pairs AS (
    SELECT 
        c1.category as cat_1,
        c2.category as cat_2,
        a.fund_name as fund_1,
        b.fund_name as fund_2,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name < b.fund_name
    JOIN fund_cat c1 ON a.fund_name = c1.fund_name
    JOIN fund_cat c2 ON b.fund_name = c2.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
        AND c1.category != c2.category
    GROUP BY a.fund_name, b.fund_name
)
SELECT 
    cat_1 || ' + ' || cat_2 as combination,
    ROUND(AVG(overlap), 2) as avg_overlap,
    ROUND(MIN(overlap), 2) as min_overlap,
    COUNT(*) as pairs
FROM pairs
GROUP BY cat_1, cat_2
ORDER BY avg_overlap ASC
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

          combination  avg_overlap  min_overlap  pairs
      Mid Cap + Index         3.18         0.71     30
Small Cap + Flexi Cap         3.30         0.27     53
Small Cap + Large Cap         3.47         0.34     46
    Small Cap + Index         3.88         0.88     24
      Index + Mid Cap         3.95         0.71     20
Flexi Cap + Small Cap         4.80         0.33     42
Large Cap + Small Cap         5.02         0.38     41
  Small Cap + Mid Cap         5.06         0.14     54
    Index + Small Cap         6.41         0.88     11
  Mid Cap + Small Cap         6.60         0.63     43
  Mid Cap + Large Cap         6.62         1.96     43
  Large Cap + Mid Cap         7.12         2.05     57
  Mid Cap + Flexi Cap         7.43         0.12     47
  Flexi Cap + Mid Cap        10.10         0.09     51
Large Cap + Flexi Cap        35.09        16.37     49
    Index + Flexi Cap        35.71        20.49     19
Flexi Cap + Large Cap        40.63        19.95     51
    Flexi 

### Hidden gems — high conviction stocks held by very few funds
These are stocks where a fund manager has placed a big bet (>2% weight) but very few other funds hold it. These represent unique, differentiated positions — potentially the fund manager's best ideas.

In [20]:
query = """
SELECT 
    stock_name,
    COUNT(DISTINCT fund_name) as num_funds,
    ROUND(MAX(holding_pct), 2) as max_weight,
    GROUP_CONCAT(fund_name, ' | ') as held_by
FROM holdings
WHERE holding_pct > 2
GROUP BY stock_name
HAVING num_funds <= 3
ORDER BY max_weight DESC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                            stock_name  num_funds  max_weight                                                                          held_by
                TVS Motor Company Ltd.          1        9.68                                                   ICICI Prudential Flexicap Fund
               Nifty 50 : Futures Near          3        8.93   Motilal Oswal Midcap Fund | Axis Small Cap Fund | Motilal Oswal Flexi Cap Fund
            One 97 Communications Ltd.          1        7.64                                                        Motilal Oswal Midcap Fund
           Kalyan Jewellers India Ltd.          3        7.48 Motilal Oswal Midcap Fund | Sundaram Mid Cap Fund | Motilal Oswal Flexi Cap Fund
CG Power and Industrial Solutions Ltd.          1        7.05                                                     Motilal Oswal Flexi Cap Fund
  Power Grid Corporation of India Ltd.          2        6.92                                Parag Parikh Flexi Cap Fund | HDFC Flexi Cap Fund

### Which fund house (AMC) dominates each sector?
Different AMCs may have sector biases based on their research teams' expertise. This reveals which AMC is the heaviest investor in each sector across all their funds.

In [21]:
query = """
WITH fund_amc AS (
    SELECT fund_name,
        CASE
            WHEN fund_name LIKE '%HDFC%' THEN 'HDFC'
            WHEN fund_name LIKE '%SBI%' THEN 'SBI'
            WHEN fund_name LIKE '%ICICI%' THEN 'ICICI'
            WHEN fund_name LIKE '%Axis%' THEN 'Axis'
            WHEN fund_name LIKE '%Kotak%' THEN 'Kotak'
            WHEN fund_name LIKE '%Nippon%' THEN 'Nippon'
            WHEN fund_name LIKE '%Mirae%' THEN 'Mirae'
            WHEN fund_name LIKE '%Motilal%' THEN 'Motilal Oswal'
            WHEN fund_name LIKE '%DSP%' THEN 'DSP'
            WHEN fund_name LIKE '%UTI%' THEN 'UTI'
            WHEN fund_name LIKE '%Parag%' THEN 'PPFAS'
            WHEN fund_name LIKE '%Canara%' THEN 'Canara Robeco'
            WHEN fund_name LIKE '%Aditya%' THEN 'Aditya Birla'
            WHEN fund_name LIKE '%Franklin%' THEN 'Franklin'
            WHEN fund_name LIKE '%Tata%' THEN 'Tata'
            WHEN fund_name LIKE '%Bandhan%' THEN 'Bandhan'
            WHEN fund_name LIKE '%Quant%' THEN 'Quant'
            WHEN fund_name LIKE '%HSBC%' THEN 'HSBC'
            WHEN fund_name LIKE '%Edelweiss%' THEN 'Edelweiss'
            WHEN fund_name LIKE '%Sundaram%' THEN 'Sundaram'
            ELSE 'Other'
        END as amc
    FROM holdings
    GROUP BY fund_name
)
SELECT 
    h.sector,
    fa.amc,
    ROUND(SUM(h.holding_pct), 2) as total_sector_weight,
    COUNT(DISTINCT h.stock_name) as stocks_in_sector
FROM holdings h
JOIN fund_amc fa ON h.fund_name = fa.fund_name
WHERE h.holding_pct > 0
GROUP BY h.sector, fa.amc
ORDER BY h.sector, total_sector_weight DESC
"""

result = pd.read_sql_query(query, conn)
# Show top AMC per sector
top_per_sector = result.groupby('sector').first().reset_index()
print(top_per_sector[['sector', 'amc', 'total_sector_weight']].head(20).to_string(index=False))

                                                                 sector           amc  total_sector_weight
                                                           2/3 wheelers         ICICI                15.20
                                                   Abrasives & bearings           SBI                 6.13
                                           Advertising & media agencies        Nippon                 0.01
                                                    Aerospace & defense         Kotak                10.28
                                                                Airline          HDFC                 5.76
                                             Airport & airport services  Aditya Birla                 1.04
                                                              Aluminium           SBI                 4.43
                                      Aluminium, copper & zinc products       Bandhan                 0.24
                                     

### Large Cap Overlap Matrix — how similar are large cap funds to each other?
A detailed pairwise comparison of all 10 large cap funds. This is the data you'd need to decide: "Should I switch from one large cap fund to another, or are they basically the same?"

In [22]:
query = """
SELECT 
    a.fund_name as fund_1,
    b.fund_name as fund_2,
    COUNT(DISTINCT a.stock_name) as common_stocks,
    ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as overlap_pct
FROM holdings a
JOIN holdings b 
    ON a.stock_name = b.stock_name
    AND a.fund_name < b.fund_name
WHERE a.holding_pct > 0 AND b.holding_pct > 0
    AND a.fund_name LIKE '%Large Cap%'
    AND b.fund_name LIKE '%Large Cap%'
GROUP BY a.fund_name, b.fund_name
ORDER BY overlap_pct DESC
"""

result = pd.read_sql_query(query, conn)
print("=== LARGE CAP OVERLAP MATRIX ===")
print(result.to_string(index=False))

=== LARGE CAP OVERLAP MATRIX ===
                              fund_1                          fund_2  common_stocks  overlap_pct
                 Axis Large Cap Fund    Canara Robeco Large Cap Fund             39        72.71
        Canara Robeco Large Cap Fund            Kotak Large Cap Fund             30        67.45
        Canara Robeco Large Cap Fund     Nippon India Large Cap Fund             33        64.00
                 Axis Large Cap Fund            Kotak Large Cap Fund             30        63.78
        Canara Robeco Large Cap Fund              UTI Large Cap Fund             33        63.24
Aditya Birla Sun Life Large Cap Fund    Canara Robeco Large Cap Fund             36        62.82
        Canara Robeco Large Cap Fund      Mirae Asset Large Cap Fund             30        62.61
Aditya Birla Sun Life Large Cap Fund              UTI Large Cap Fund             32        61.55
Aditya Birla Sun Life Large Cap Fund            Kotak Large Cap Fund             31        60.

### Mid-conviction stocks — the "sweet spot" between popular and hidden
Not the mega popular stocks (held by 25+ funds) and not the hidden gems (held by 1-3). These stocks are held by 5-15 funds with decent weight — popular enough to have fund manager consensus, but not so crowded that everyone owns them.

In [23]:
query = """
SELECT 
    stock_name,
    sector,
    COUNT(DISTINCT fund_name) as num_funds,
    ROUND(AVG(holding_pct), 2) as avg_weight,
    ROUND(SUM(holding_pct), 2) as total_weight,
    GROUP_CONCAT(DISTINCT market_cap) as cap_categories
FROM holdings
WHERE holding_pct > 0
GROUP BY stock_name
HAVING num_funds >= 5 AND num_funds <= 15
ORDER BY avg_weight DESC
LIMIT 20
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                                 stock_name                               sector  num_funds  avg_weight  total_weight cap_categories
                    Nifty 50 : Futures Near           Exchange and data platform          5        3.85         19.26          Other
                Kalyan Jewellers India Ltd.          Gems, jewellery and watches          5        3.85         19.25          Other
                     Fortis Healthcare Ltd.                             Hospital          8        2.93         23.41          Other
                          Federal Bank Ltd.                  Private sector bank          9        2.80         25.17        Mid Cap
                                Indian Bank                   Public sector bank          7        2.40         16.80        Mid Cap
                        KEI Industries Ltd.                 Cables - electricals          8        2.34         18.70      Small Cap
                               Coforge Ltd.    Computers - software &

### Diversification Score — which fund is the most unique?
For each fund, I calculated its average overlap with every other fund in the database. Lower score = more unique portfolio. This helps identify truly differentiated funds vs. funds that look like index clones.

In [24]:
query = """
WITH pair_overlaps AS (
    SELECT 
        a.fund_name,
        b.fund_name as other_fund,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name != b.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
    GROUP BY a.fund_name, b.fund_name
)
SELECT 
    fund_name,
    ROUND(AVG(overlap), 2) as diversification_score
FROM pair_overlaps
GROUP BY fund_name
ORDER BY diversification_score ASC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print("Lower score = More unique fund")
print(result.to_string(index=False))

Lower score = More unique fund
                    fund_name  diversification_score
       Bandhan Small Cap Fund                   4.19
          HDFC Small Cap Fund                   4.60
           SBI Small Cap Fund                   4.61
         Quant Small Cap Fund                   4.83
           DSP Small Cap Fund                   4.87
Franklin India Small Cap Fund                   5.73
         Kotak Small Cap Fund                   7.20
              SBI Midcap Fund                   7.27
          Axis Small Cap Fund                   7.34
          HSBC Small Cap Fund                   7.47
            HDFC Mid Cap Fund                   8.77
      Mirae Asset Midcap Fund                   8.78
  Nippon India Small Cap Fund                   9.44
        Sundaram Mid Cap Fund                   9.72
    Motilal Oswal Midcap Fund                  10.35


### Market cap distribution across fund categories
Are "mid cap" funds actually investing in mid caps? Or are they sneaking in large caps for safety? This checks if funds are staying true to their SEBI-mandated category label.

In [25]:
query = """
WITH fund_cat AS (
    SELECT fund_name,
        CASE 
            WHEN fund_name LIKE '%Large Cap%' THEN 'Large Cap'
            WHEN fund_name LIKE '%Mid Cap%' OR fund_name LIKE '%Midcap%' THEN 'Mid Cap'
            WHEN fund_name LIKE '%Small Cap%' THEN 'Small Cap'
            WHEN fund_name LIKE '%Flexi%' OR fund_name LIKE '%Flexicap%' THEN 'Flexi Cap'
            WHEN fund_name LIKE '%Index%' OR fund_name LIKE '%Sensex%' OR fund_name LIKE '%Nifty%' THEN 'Index'
            ELSE 'Other'
        END as category
    FROM holdings
    GROUP BY fund_name
)
SELECT 
    fc.category,
    h.market_cap,
    COUNT(DISTINCT h.stock_name) as unique_stocks,
    ROUND(AVG(h.holding_pct), 2) as avg_weight,
    ROUND(SUM(h.holding_pct), 2) as total_weight
FROM holdings h
JOIN fund_cat fc ON h.fund_name = fc.fund_name
WHERE h.holding_pct > 0 AND h.market_cap != ''
GROUP BY fc.category, h.market_cap
ORDER BY fc.category, total_weight DESC
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

 category market_cap  unique_stocks  avg_weight  total_weight
Flexi Cap  Large Cap             82        2.14        437.14
Flexi Cap      Other            135        1.50        326.43
Flexi Cap    Mid Cap             62        1.17        116.11
Flexi Cap  Small Cap             65        0.76         57.83
    Index  Large Cap             33        2.29        352.38
    Index      Other             12        2.39        128.94
    Index    Mid Cap              6        0.84         18.47
Large Cap  Large Cap             77        1.99        622.50
Large Cap      Other             72        1.46        259.00
Large Cap    Mid Cap             51        0.79         69.25
Large Cap  Small Cap             24        0.59         17.05
  Mid Cap    Mid Cap             90        1.50        387.44
  Mid Cap      Other            110        1.29        315.72
  Mid Cap  Small Cap             62        1.21        146.62
  Mid Cap  Large Cap             42        1.38        113.02
Small Ca

### Optimal 5-fund portfolio with minimum overlap
The final practical output — if you could only hold 5 funds (one from each category), which combination gives you maximum diversification? I picked the most "unique" fund from each category based on the diversification score.

In [26]:
query = """
WITH fund_cat AS (
    SELECT fund_name,
        CASE 
            WHEN fund_name LIKE '%Large Cap%' THEN 'Large Cap'
            WHEN fund_name LIKE '%Mid Cap%' OR fund_name LIKE '%Midcap%' THEN 'Mid Cap'
            WHEN fund_name LIKE '%Small Cap%' THEN 'Small Cap'
            WHEN fund_name LIKE '%Flexi%' OR fund_name LIKE '%Flexicap%' THEN 'Flexi Cap'
            WHEN fund_name LIKE '%Index%' OR fund_name LIKE '%Sensex%' OR fund_name LIKE '%Nifty%' THEN 'Index'
            ELSE 'Other'
        END as category
    FROM holdings
    GROUP BY fund_name
),
pair_overlaps AS (
    SELECT 
        a.fund_name,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name != b.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
    GROUP BY a.fund_name, b.fund_name
),
fund_scores AS (
    SELECT 
        fund_name,
        ROUND(AVG(overlap), 2) as avg_overlap_score
    FROM pair_overlaps
    GROUP BY fund_name
)
SELECT 
    fc.category,
    fs.fund_name,
    fs.avg_overlap_score,
    COUNT(DISTINCT h.stock_name) as total_stocks
FROM fund_scores fs
JOIN fund_cat fc ON fs.fund_name = fc.fund_name
JOIN holdings h ON fs.fund_name = h.fund_name AND h.holding_pct > 0
GROUP BY fc.category, fs.fund_name
ORDER BY fc.category, fs.avg_overlap_score ASC
"""

result = pd.read_sql_query(query, conn)
best = result.groupby('category').first().reset_index()
print("=== BEST 5-FUND PORTFOLIO (Minimum Overlap) ===\n")
print(best.to_string(index=False))

=== BEST 5-FUND PORTFOLIO (Minimum Overlap) ===

 category                    fund_name  avg_overlap_score  total_stocks
Flexi Cap Motilal Oswal Flexi Cap Fund              15.53            23
    Index   HDFC BSE Sensex Index Fund              34.31            30
Large Cap           SBI Large Cap Fund              25.19            45
  Mid Cap              SBI Midcap Fund               7.27            51
Small Cap       Bandhan Small Cap Fund               4.19           244


## Key Findings

### 1. Index Funds Are Nearly Identical
Nifty 50 index funds from HDFC, SBI, UTI share ~100% portfolio overlap. Holding more than one Nifty 50 index fund is completely pointless.

### 2. Large Cap Funds Have 54% Average Overlap
More than half the portfolio is the same across large cap funds. This happens because SEBI mandates 80%+ investment in just ~100 stocks — fund managers have very little room to be different.

### 3. Small Cap Funds Are the Most Unique
Only 11% average overlap across small cap funds. With 1000+ stocks to choose from, each fund manager builds a genuinely different portfolio. Best category for true diversification.

### 4. HDFC Bank and ICICI Bank Dominate Indian Mutual Funds
Present in 28 out of 45 funds with ~7-8% average weight each. Most mutual fund investors are heavily exposed to banking sector risk without realizing it.

### 5. Eternal Ltd. (Zomato) Is the Most Widely Held Stock
Found in 31 out of 45 funds — more than HDFC Bank or Reliance. The new-age tech stock has become a mutual fund darling across all categories.

### 6. Private Sector Banking Is the Most Crowded Sector
Total weight of 665 across all funds — nearly 3x the second most crowded sector (IT at 240). A banking crisis would hit almost every mutual fund investor in India simultaneously.

### 7. Index Funds Have Zero Small Cap Exposure
Nifty 50 and Sensex trackers contain absolutely no small cap stocks. Investors relying solely on index funds are completely missing out on the small cap segment of the market.

### 8. Holding Two Large Cap Funds Gives You 37% in Just Two Stocks
If you hold both HDFC Large Cap and ICICI Large Cap equally, nearly 19% goes into ICICI Bank and 18% into HDFC Bank. That's 37% of your money in just two banking stocks.

### 9. Parag Parikh Flexi Cap Is the Most Unique Active Fund
Its foreign holdings (Alphabet, Meta, Amazon, Microsoft) ensure minimal overlap with any other Indian equity fund. No other fund in our database holds US tech stocks.

### 10. Motilal Oswal Midcap and SBI Midcap Have Zero Overlap
Despite both being mid cap funds, they share absolutely no common stocks. Motilal Oswal runs a 26-stock high-conviction portfolio while SBI holds 51 stocks — proof that same category doesn't mean same portfolio.

### 11. Flexi Cap Funds Aren't Really "Flexi"
Flexi Cap funds have 437 total weight in large caps vs only 58 in small caps — a 7.5x skew. Despite being allowed to invest anywhere, fund managers overwhelmingly prefer large caps.

### 12. Don't Pair Large Cap with Index Funds
61.7% average overlap — the worst category combination. Large cap funds essentially hold the same stocks as Nifty 50, so holding both adds very little diversification.

### 13. Best Diversification Strategy
Pair a Mid Cap or Small Cap fund with an Index Fund — only 3-4% overlap. You get market-wide exposure from the index fund plus genuine diversification from the smaller cap fund.

### 14. The Optimal 5-Fund Portfolio
For maximum diversification, hold one fund from each category with the lowest overlap score:
- **Flexi Cap:** Motilal Oswal Flexi Cap Fund (23 stocks)
- **Index:** HDFC BSE Sensex Index Fund (30 stocks)
- **Large Cap:** SBI Large Cap Fund (45 stocks)
- **Mid Cap:** SBI Midcap Fund (51 stocks)
- **Small Cap:** Bandhan Small Cap Fund (244 stocks)

## Limitations
- Data sourced from Moneycontrol.com which may have slight discrepancies with official AMC portfolio disclosures
- Holdings are as of a single date (Feb/Mar 2026) — overlap changes monthly as fund managers rebalance their portfolios
- Analysis covers equity holdings only — debt, cash, and other positions are excluded
- "Other" market cap category includes foreign stocks, recently listed companies, and boundary cases not classified by Moneycontrol

## Tools & Technologies Used
- **Python** — requests, BeautifulSoup (web scraping), pandas (data manipulation)
- **SQL (SQLite)** — database design, complex queries with JOINs, CTEs, window functions, aggregations
- **Data Source** — Moneycontrol.com (publicly available mutual fund portfolio holdings)